## Tool Use

**Tool use** (also called *function calling*) lets Claude go beyond generating text — it can request that a piece of code on your side be run, then use that result to continue answering.

Claude itself never executes anything. The flow is always:

1. **You** describe one or more tools to Claude (a name, description, and JSON schema of the inputs it accepts).
2. **Claude** decides, based on the user's message, whether a tool is needed — and if so, replies with a `tool_use` block containing the tool's name and the arguments it wants to call it with.
3. **You** run the corresponding function yourself, with those arguments.
4. **You** send the function's output back to Claude as a `tool_result` block.
5. **Claude** uses that result to write its final answer (or asks for another tool call, and the loop continues).

This is why tool use matters: it's how Claude reaches outside its own knowledge and training data to do things like check the current time, query a database, call an API, edit a file, or search the web — anything you can wrap in a function.

There are two categories of tools:

- **Custom tools** — a schema you define yourself, backed by your own Python function.
- **Built-in tools** — tools Anthropic already implements on Claude's side, like the text editor tool and web search tool. You only declare that you want to use them; for some (like web search) Claude runs them directly, while for others (like the text editor) Claude still asks you to execute the actual file operation.


In [1]:
%%capture
%pip install anthropic python-dotenv

In [2]:
# Setup: load environment variables and create the client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

## 1. Defining a Tool Function

**Definition:** A tool function is ordinary Python code that performs the actual work a tool represents — Claude never sees or runs this code directly. It only ever sees the *schema* we'll define in the next section; the function itself lives entirely on our side.

Every tool starts life as a plain function, exactly as you'd write it for any other purpose:

- It takes normal Python arguments.
- It returns a normal Python value (usually a string, or something JSON-serializable).
- It can raise exceptions — we'll see in section 5 how those get reported back to Claude as tool errors instead of crashing the program.

Our running example: a function that returns the current date/time in a given format. There's nothing Claude-specific about it yet — we can call and test it like any other function.


In [3]:
from datetime import datetime


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


# Try it directly, with no Claude involved yet
get_current_datetime("%H:%M:%S")


'18:33:31'

## 2. Defining a Tool Schema

**Definition:** A tool schema is a JSON description we hand to Claude in the `tools` parameter of an API call. It tells Claude that a tool exists, what it's for, and exactly what arguments it accepts — Claude reads only this description; it never sees our function's source code.

A schema has three parts:

- **`name`** — a unique identifier Claude will use to refer to this tool (must match what we check for later, in `run_tool`).
- **`description`** — plain-English text explaining what the tool does and when to use it. This is what Claude actually reasons over to decide *whether* to call the tool, so being specific here matters as much as writing good code.
- **`input_schema`** — a JSON Schema object listing each parameter's name, type, description, and which ones are `required`. Claude uses this to construct valid arguments.

The schema and the function are two separate things that must stay in sync by convention — the API does not check that they match.


In [4]:
from anthropic.types import ToolParam

get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": "Returns the current date and time formatted according to the specified format string.",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": "A Python strftime format string, e.g. '%H:%M:%S' for just the time.",
                    "default": "%Y-%m-%d %H:%M:%S",
                }
            },
            "required": [],
        },
    }
)


## 3. Handling Message Blocks

**Definition:** A Claude response is not a single string — its `content` is a *list of blocks*, each with a `type`. When tools are involved, a response can contain a `text` block, a `tool_use` block, or both, and we have to inspect `response.content` to see which.

Key things to know about this response:

- **`stop_reason`** tells us *why* Claude stopped generating. When it's `"tool_use"`, Claude is pausing specifically because it wants a tool run before it can continue — this is the signal our code checks for.
- A **`tool_use` block** carries three fields we need: `id` (a unique identifier for this specific call, used to match up the result later), `name` (which tool Claude wants), and `input` (the arguments, already validated against our schema).

Now let's give Claude the tool and ask a question that requires it. Claude's response won't contain the answer directly — it will contain a `tool_use` content block asking us to run the tool on its behalf.


In [ ]:
messages = [
    {"role": "user", "content": "What is the exact time, formatted as HH:MM:SS?"}
]

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

response  
# response.stop_reason # "tool_use" tells us Claude wants to call a tool

Message(id='msg_011CeMcz6yoqqbd2tRwryYg3', container=None, content=[ToolUseBlock(id='toolu_01QqasaXMhr6RiMY871MDrgK', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=624, output_tokens=62, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [7]:
# Inspect the tool_use block Claude produced
tool_use_block = response.content[-1]
tool_use_block


ToolUseBlock(id='toolu_01KfuXhCsKnXE7aAVzNVk7tg', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)

## 4. Sending Tool Results

**Definition:** A tool result is how we hand a tool's output back to Claude — sent as a `user`-role message containing a `tool_result` block, not a plain string.

Two rules make this work correctly:

- **`tool_use_id` must match** the `id` from the `tool_use` block we're responding to. This is how Claude links a result back to the specific call it made — critical when multiple tools are called in one turn.
- **The full conversation history matters.** Before sending the result, we must also append Claude's own `tool_use` turn (`response.content`) to `messages`. The API expects the exact sequence: user asks → assistant requests a tool → user supplies the result → assistant answers.

We run the requested tool ourselves, then send the result back to Claude as a `tool_result` block so it can finish answering.


In [8]:
# Run the tool locally, using the arguments Claude provided
result = get_current_datetime(**tool_use_block.input)
result


'17:03:27'

In [9]:
# Add Claude's tool_use turn, then our tool_result turn
messages.append({"role": "assistant", "content": response.content})
messages.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use_block.id,
                "content": result,
                "is_error": False,
            }
        ],
    }
)

final_response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

final_response.content[0].text


'The exact time is **17:03:27**.'

## 5. Multi-Turn Conversations with Tools

**Definition:** A multi-turn tool conversation is what happens when the single round-trip from section 4 isn't enough — Claude may need to call a tool, look at the result, and then call a tool *again* (the same one or a different one) before it can give a final answer. This happens automatically whenever a question can't be resolved in one tool call.

Doing this by hand doesn't scale, so we generalize the section 3-4 pattern into a loop:

- **`run_tool`** dispatches by name to the right Python function — the bridge between a `tool_use` block's `name` and the function that actually implements it.
- **`run_tools`** handles *all* `tool_use` blocks in a single response (there can be more than one — more on that in section 6), and wraps each function's output — or, if it raised, the error — into a `tool_result` block.
- **`run_conversation`** is the loop itself: call Claude, hand back any tool results, and repeat until `stop_reason` is no longer `"tool_use"`.

This loop is the core pattern behind every agentic use of Claude — sections 6 onward just add more tools and options on top of it.


In [10]:
import json


def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, message):
    messages.append({"role": "assistant", "content": message.content})


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")


def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(message):
    tool_result_blocks = []
    for block in message.content:
        if block.type != "tool_use":
            continue
        try:
            output = run_tool(block.name, block.input)
            tool_result_blocks.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(output),
                    "is_error": False,
                }
            )
        except Exception as e:
            tool_result_blocks.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": f"Error: {e}",
                    "is_error": True,
                }
            )
    return tool_result_blocks


def run_conversation(messages, tools):
    while True:
        response = client.messages.create(
            model=model, max_tokens=1000, messages=messages, tools=tools
        )
        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        add_user_message(messages, run_tools(response))

    return messages


In [11]:
messages = []
add_user_message(
    messages,
    "What is the current time in HH:MM format? Also, what is the current time in SS format?",
)

run_conversation(messages, tools=[get_current_datetime_schema])



The current time in HH:MM format is **17:03**.

The current seconds (SS format) is **35**.


[{'role': 'user',
  'content': 'What is the current time in HH:MM format? Also, what is the current time in SS format?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_014UE4nuZGA5Ze7C7JuwJzYG', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M'}, name='get_current_datetime', type='tool_use', toolset_name=None),
   ToolUseBlock(id='toolu_01TZXAjX5hHNfhJC3DQDW9TB', caller=DirectCaller(type='direct'), input={'date_format': '%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_014UE4nuZGA5Ze7C7JuwJzYG',
    'content': '"17:03"',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01TZXAjX5hHNfhJC3DQDW9TB',
    'content': '"35"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The current time in HH:MM format is **17:03**.\n\nThe current seconds (SS format) is **35**.', type='text')]}]

## 6. Multiple Tools

**Definition:** The `tools` list passed to `client.messages.create` isn't limited to one entry — Claude can be given a whole toolbox at once, and it independently decides which tool (or tools) are relevant to the user's request.

This changes what `run_tool` and `run_tools` need to handle:

- **`run_tool` becomes a dispatcher** — it must check `tool_name` and route to the matching function, since more than one tool is now possible.
- **A single response can request multiple tool calls at once** — e.g. Claude might emit two `tool_use` blocks in the same turn if it can tell upfront it needs both. `run_tools` already handles this, since it loops over every `tool_use` block in the message.
- **Claude can also chain tools across turns** — call one tool, see the result, then decide it needs a *different* tool next. This is what makes the multi-turn loop from section 5 essential once more than one tool is in play.

We add a second, related tool — `add_duration_to_datetime` — so answering the question requires calling both tools in sequence: first get the current time, then add a duration to it.


In [12]:
from datetime import timedelta


def add_duration_to_datetime(datetime_str, duration=0, unit="days", input_format="%Y-%m-%d %H:%M:%S"):
    date = datetime.strptime(datetime_str, input_format)
    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")
    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


add_duration_to_datetime_schema = ToolParam(
    {
        "name": "add_duration_to_datetime",
        "description": "Adds a duration (seconds, minutes, hours, or days) to a datetime string and returns the resulting datetime.",
        "input_schema": {
            "type": "object",
            "properties": {
                "datetime_str": {"type": "string", "description": "The starting datetime, formatted as '%Y-%m-%d %H:%M:%S'."},
                "duration": {"type": "number", "description": "Amount of time to add. Can be negative."},
                "unit": {"type": "string", "description": "One of: seconds, minutes, hours, days."},
            },
            "required": ["datetime_str", "duration", "unit"],
        },
    }
)


In [13]:
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    raise ValueError(f"Unknown tool: {tool_name}")


In [14]:
messages = []
add_user_message(messages, "What will the time be 90 minutes from now?")

run_conversation(
    messages,
    tools=[get_current_datetime_schema, add_duration_to_datetime_schema],
)


I'll help you find out what time it will be 90 minutes from now.

The time 90 minutes from now will be **6:33:48 PM** (18:33:48).


[{'role': 'user', 'content': 'What will the time be 90 minutes from now?'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you find out what time it will be 90 minutes from now.", type='text'),
   ToolUseBlock(id='toolu_01Dj5qTtvRkCyvsCKhtcGVKG', caller=DirectCaller(type='direct'), input={}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01Dj5qTtvRkCyvsCKhtcGVKG',
    'content': '"2026-08-24 17:03:48"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01LGfqP1JX1jxKevGG6wW2Ys', caller=DirectCaller(type='direct'), input={'datetime_str': '2026-08-24 17:03:48', 'duration': 90, 'unit': 'minutes'}, name='add_duration_to_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01LGfqP1JX1jxKevGG6wW2Ys',
    'content': '"Monday, August 24, 2026 06:33

## 7. Fine-Grained Tool Calling

**Definition:** Fine-grained tool calling is a streaming mode (enabled via the `fine-grained-tool-streaming-2025-05-14` beta header) where Claude sends a tool call's JSON input incrementally, as it's generated, instead of only after the whole thing is ready. Combined with `tool_choice`, it also covers how we can constrain *which* tool Claude is allowed to pick.

Two related but separate concepts here:

- **Streaming tool input** — normally we wait for the full response before we can see a tool call. With `client.beta.messages.stream(...)`, we instead get a sequence of chunks: `content_block_start` (a new block, e.g. a `tool_use`, is beginning), `input_json` with `partial_json` (fragments of the arguments' JSON as they're produced), and `content_block_stop`. This is useful for showing a live "Claude is filling in the form..." UI instead of a silent wait.
- **`tool_choice`** — by default (`"auto"`), Claude decides for itself whether and which tool to call. Setting `tool_choice={"type": "tool", "name": "..."}` forces Claude to call that exact tool, which is useful when we already know from context that a tool call is required and want to skip Claude's own deliberation.

Since this needs a tool with a bit more structure to make streaming visible, we use a small `save_note` tool as a focused example.


In [15]:
save_note_schema = ToolParam(
    {
        "name": "save_note",
        "description": "Saves a short note with a title and body",
        "input_schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string", "description": "Short title for the note"},
                "body": {"type": "string", "description": "The note content, 2-3 sentences"},
            },
            "required": ["title", "body"],
        },
    }
)


def save_note(**kwargs):
    return "Note saved!"


In [16]:
messages = [{"role": "user", "content": "Save a short note reminding me to renew my passport."}]

with client.beta.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[save_note_schema],
    tool_choice={"type": "tool", "name": "save_note"},
    betas=["fine-grained-tool-streaming-2025-05-14"],
) as stream:
    for chunk in stream:
        if chunk.type == "content_block_start" and chunk.content_block.type == "tool_use":
            print(f'>>> Tool call: "{chunk.content_block.name}"')
        if chunk.type == "input_json" and chunk.partial_json:
            print(chunk.partial_json, end="")

    # response = stream.get_final_message()


>>> Tool call: "save_note"
{"title": "Renew Passport", "body": "Remember to renew your passport before it expires. Check the expiration date and start the renewal process early to avoid any travel disruptions."}

## 8. Web Search Tool

**Definition:** `web_search` is another built-in tool — this one is **server-executed**. Claude performs the search itself against the live web; we never receive a `tool_use` block to handle ourselves, and there's no local function or `run_tool` case to write.

Key configuration options:

- **`max_uses`** — caps how many searches Claude can perform while answering a single request, to bound cost and latency.
- **`allowed_domains`** (or its counterpart `blocked_domains`) — restricts results to a trusted set of sources, which matters whenever source credibility affects the answer's reliability — health, legal, financial, or in this case a specific weather provider.
- **The response already contains the synthesized answer** — Claude reads the search results and writes the final text for us; we just call `client.messages.create` once and read `response.content`, no result-sending loop required.

Another built-in tool: `web_search`. Claude runs the search itself (no local function needed) and incorporates the results into its answer. Here we restrict it to `www.accuweather.com` and ask for the current weather in Bengaluru.

In [20]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["www.accuweather.com"],
}

messages = []
add_user_message(messages, "What is the current weather in Bengaluru?")

response = client.messages.create(
    model=model, max_tokens=1000, messages=messages, tools=[web_search_schema]
)

text_from_message(response)

'Based on the current weather information for Bengaluru:\n\n\nBengaluru is currently cloudy with a temperature of 81°F\n. \nThe current conditions include 66% humidity, 91% cloud cover, a UV index of 0, and winds of 14 mph\n.\n\nThe forecast for today indicates \ncloudy conditions with a thunderstorm in spots this afternoon\n.'